# LAB 05 - Bài 15: Baseline và Simple RNN

Notebook so sánh **TF-IDF + Logistic Regression** với **Embedding + Simple RNN** trên cùng bộ ACL IMDb train/test.

Chạy các code cell theo thứ tự. Cell kết quả lưu bảng metrics thành `/content/lab05_bai15_results.csv`. Thí nghiệm xáo trộn token trên test giúp quan sát độ nhạy với word order; đây chỉ là một phép kiểm tra thực nghiệm.

Lần chạy đầu tải dataset khoảng 84 MB. Simple RNN huấn luyện trên toàn bộ train set nên có thể mất vài phút; đổi `EPOCHS` nếu cần.

In [1]:
# ============================================================
# BÀI 15 - EXPERIMENT 3: CHUẨN BỊ DỮ LIỆU ACL IMDb
# Dataset chính thức của Stanford: train/test, neg=0, pos=1.
# ============================================================

from pathlib import Path
import tarfile
import urllib.request
import numpy as np
import pandas as pd

DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
DATA_DIR = Path("/content/aclImdb")
ARCHIVE_PATH = Path("/content/aclImdb_v1.tar.gz")

# Tải và giải nén một lần trong mỗi runtime Colab.
if not (DATA_DIR / "train" / "pos").exists():
    if not ARCHIVE_PATH.exists():
        print("Đang tải ACL IMDb từ Stanford (~84 MB)...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    print("Đang giải nén dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path="/content")

def read_split(split):
    texts, labels = [], []
    for label_name, label_id in (("neg", 0), ("pos", 1)):
        folder = DATA_DIR / split / label_name
        for file_path in sorted(folder.glob("*.txt")):
            texts.append(file_path.read_text(encoding="utf-8", errors="replace").strip())
            labels.append(label_id)
    return texts, np.asarray(labels, dtype=np.int32)

train_texts, y_train = read_split("train")
test_texts, y_test = read_split("test")

print("Train samples:", len(train_texts))
print("Test samples:", len(test_texts))
print("Train label counts (0=neg, 1=pos):", np.bincount(y_train))
print("Test label counts (0=neg, 1=pos):", np.bincount(y_test))


Đang tải ACL IMDb từ Stanford (~84 MB)...
Đang giải nén dataset...


/tmp/ipykernel_556/1843585848.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path="/content")


Train samples: 25000
Test samples: 25000
Train label counts (0=neg, 1=pos): [12500 12500]
Test label counts (0=neg, 1=pos): [12500 12500]


In [2]:
# ============================================================
# BÀI 15 - BASELINE VÀ RNN SEQUENCE CLASSIFIER
# So sánh TF-IDF + Logistic Regression với Embedding + Simple RNN.
# ============================================================

import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

MAX_VOCAB_SIZE = 20_000
MAX_LEN = 200
EMBEDDING_DIM = 64
HIDDEN_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 3
SEED = 42

# Hàm tính các metrics theo lớp positive=1.
def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

results = []

# -------------------- Baseline: TF-IDF + Logistic Regression --------------------
# Dùng unigram để biểu diễn theo bag-of-words, không giữ thứ tự từ.
vectorizer = TfidfVectorizer(max_features=MAX_VOCAB_SIZE, ngram_range=(1, 1))
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

logreg = LogisticRegression(max_iter=1000, random_state=SEED)
start_time = time.perf_counter()
logreg.fit(X_train_tfidf, y_train)
logreg_train_seconds = time.perf_counter() - start_time

pred_logreg = logreg.predict(X_test_tfidf)
results.append({
    "Model": "TF-IDF + Logistic Regression",
    "Training time (seconds)": logreg_train_seconds,
    **get_metrics(y_test, pred_logreg),
})

# Minh họa: unigram TF-IDF không đổi khi chỉ đảo thứ tự cùng các từ.
example_a = "the movie is not good"
example_b = "good not movie is the"
example_vectors = vectorizer.transform([example_a, example_b])
print("TF-IDF vectors unchanged after word reordering:",
      (example_vectors[0] != example_vectors[1]).nnz == 0)

# -------------------- Sequence model: Embedding + Simple RNN --------------------
# Fit vocabulary chỉ trên train. 0 được dành cho padding.
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

X_train_seq = pad_sequences(
    train_sequences, maxlen=MAX_LEN, padding="post", truncating="post", value=0
)
X_test_seq = pad_sequences(
    test_sequences, maxlen=MAX_LEN, padding="post", truncating="post", value=0
)

tf.keras.utils.set_random_seed(SEED)
rnn_model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True),
    layers.SimpleRNN(HIDDEN_SIZE),
    layers.Dense(1, activation="sigmoid"),
])
rnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

start_time = time.perf_counter()
rnn_model.fit(
    X_train_seq,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
rnn_train_seconds = time.perf_counter() - start_time

prob_rnn = rnn_model.predict(X_test_seq, batch_size=BATCH_SIZE, verbose=0).ravel()
pred_rnn = (prob_rnn >= 0.5).astype(np.int32)
results.append({
    "Model": "Embedding + Simple RNN",
    "Training time (seconds)": rnn_train_seconds,
    **get_metrics(y_test, pred_rnn),
})

# Bằng chứng bổ sung về thứ tự: giữ nguyên các token nhưng xáo trộn thứ tự
# trên test. Nếu metrics đổi, đó là dấu hiệu prediction của RNN nhạy với thứ tự.
rng = np.random.default_rng(SEED)
X_test_shuffled = np.zeros_like(X_test_seq)
for i, row in enumerate(X_test_seq):
    tokens = row[row != 0].copy()
    rng.shuffle(tokens)
    X_test_shuffled[i, :len(tokens)] = tokens

prob_rnn_shuffled = rnn_model.predict(
    X_test_shuffled, batch_size=BATCH_SIZE, verbose=0
).ravel()
pred_rnn_shuffled = (prob_rnn_shuffled >= 0.5).astype(np.int32)
shuffle_metrics = get_metrics(y_test, pred_rnn_shuffled)

results_df = pd.DataFrame(results)
print("\nBảng kết quả Bài 15 (test):")
print(results_df.round(4).to_string(index=False))
print("\nRNN khi xáo trộn thứ tự token ở test (chỉ để phân tích):")
print(pd.Series(shuffle_metrics).round(4).to_string())
print("\nRNN accuracy change after shuffling:",
      round(get_metrics(y_test, pred_rnn)["Accuracy"] - shuffle_metrics["Accuracy"], 4))

# Lưu bảng kết quả để nộp hoặc đưa vào báo cáo.
OUTPUT_CSV = "/content/lab05_bai15_results.csv"
results_df.to_csv(OUTPUT_CSV, index=False)
print("\nĐã lưu bảng kết quả:", OUTPUT_CSV)

# Gợi ý diễn giải:
# - So sánh Accuracy, Precision, Recall, F1 và training time; không chỉ nhìn Accuracy.
# - TF-IDF unigram bỏ qua thứ tự từ. Simple RNN đọc token theo thứ tự.
# - Thử nghiệm đảo thứ tự là quan sát thực nghiệm, không phải bằng chứng đầy đủ
#   rằng RNN luôn học được ngữ nghĩa hoặc dependency dài.


TF-IDF vectors unchanged after word reordering: True
Epoch 1/3
352/352 - 38s - 109ms/step - accuracy: 0.6654 - loss: 0.6056 - val_accuracy: 0.8204 - val_loss: 0.4349
Epoch 2/3
352/352 - 33s - 94ms/step - accuracy: 0.7395 - loss: 0.5213 - val_accuracy: 0.5236 - val_loss: 0.7816
Epoch 3/3
352/352 - 41s - 116ms/step - accuracy: 0.8253 - loss: 0.4066 - val_accuracy: 0.7164 - val_loss: 0.6816

Bảng kết quả Bài 15 (test):
                       Model  Training time (seconds)  Accuracy  Precision  Recall     F1
TF-IDF + Logistic Regression                   0.6433    0.8836     0.8848  0.8821 0.8834
      Embedding + Simple RNN                 112.2563    0.7952     0.8510  0.7156 0.7775

RNN khi xáo trộn thứ tự token ở test (chỉ để phân tích):
Accuracy     0.7930
Precision    0.8447
Recall       0.7181
F1           0.7763

RNN accuracy change after shuffling: 0.0021

Đã lưu bảng kết quả: /content/lab05_bai15_results.csv
